# Math Foundations for Exoplanet Analysis

In [13]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = Path.cwd() / "data"
eu_df = pd.read_csv(
    DATA_DIR / "exoplanet.eu_catalog_28-05-26_22_04_09.csv", low_memory=False
)
eu_df.head()

,name,planet_status,mass,mass_error_min,mass_error_max,mass_sini,mass_sini_error_min,mass_sini_error_max,radius,radius_error_min,...,star_sp_type,star_age,star_age_error_min,star_age_error_max,star_teff,star_teff_error_min,star_teff_error_max,star_detected_disc,star_magnetic_field,star_alternate_names
0,109 Psc b,Confirmed,6.800,0.300,0.900,6.3830,0.078,0.078,1.152,NaN,...,G5 IV,6.90,0.60,0.60,5600.0,80.000,80.000,NaN,NaN,HD 10697
1,112 Psc b,Confirmed,NaN,0.005,0.004,0.0330,0.005,0.004,NaN,NaN,...,G0IV,NaN,NaN,NaN,5986.0,105.437,105.437,NaN,NaN,HD 12235
2,112 Psc c,Confirmed,9.866,1.781,3.190,NaN,NaN,NaN,NaN,NaN,...,G0IV,NaN,NaN,NaN,5986.0,105.437,105.437,NaN,NaN,HD 12235
3,11 UMi b,Confirmed,NaN,1.100,1.100,11.0873,1.100,1.100,NaN,NaN,...,K4III,1.56,0.54,0.54,4340.0,70.000,70.000,NaN,NaN,NaN
4,14 And Ab,Confirmed,NaN,0.230,0.230,4.6840,0.230,0.230,NaN,NaN,...,K0III,NaN,NaN,NaN,4813.0,20.000,20.000,NaN,NaN,NaN


## Algebra: uncertainty widths and ratios

In [14]:
def uncertainty_width(lower, upper):
    return np.abs(lower) + np.abs(upper)


math_df = eu_df[
    [
        "name",
        "mass",
        "mass_error_min",
        "mass_error_max",
        "radius",
        "orbital_period",
        "semi_major_axis",
        "star_mass",
    ]
].copy()
math_df["mass_uncertainty_width"] = uncertainty_width(
    math_df["mass_error_min"], math_df["mass_error_max"]
)
math_df["relative_mass_uncertainty"] = (
    math_df["mass_uncertainty_width"] / math_df["mass"]
)
math_df[["name", "mass", "mass_uncertainty_width", "relative_mass_uncertainty"]].head(
    10
)

,name,mass,mass_uncertainty_width,relative_mass_uncertainty
0,109 Psc b,6.800,1.200,0.176471
1,112 Psc b,NaN,0.009,NaN
2,112 Psc c,9.866,4.971,0.503852
3,11 UMi b,NaN,2.200,NaN
4,14 And Ab,NaN,0.460,NaN
5,14 Her b,8.900,3.000,0.337079
6,14 Her c,7.900,2.800,0.354430
7,16 Cyg Bb,NaN,0.160,NaN
8,18 Del Ab,NaN,0.720,NaN
9,1RXS 1609 b,6.600,0.600,0.090909


## Algebra + Calculus intuition: Kepler-inspired derived features

In [15]:
def kepler_like_scale(period_days, star_mass_solar):
    period_years = period_days / 365.25
    return np.cbrt(np.maximum(period_years**2 * star_mass_solar, 0))


math_df["kepler_like_axis"] = kepler_like_scale(
    math_df["orbital_period"], math_df["star_mass"]
)
math_df[["name", "semi_major_axis", "kepler_like_axis"]].dropna().head(10)

,name,semi_major_axis,kepler_like_axis
0,109 Psc b,2.051,2.139666
1,112 Psc b,0.054,0.054247
2,112 Psc c,22.210,22.163299
3,11 UMi b,1.540,1.531983
4,14 And Ab,0.830,0.828909
5,14 Her b,2.839,2.762295
6,14 Her c,20.000,26.378236
7,16 Cyg Bb,1.680,1.691450
8,18 Del Ab,2.600,2.571793
10,23 Lib b,0.810,0.811635


## Statistics: skew and log transforms

In [16]:
stats_df = eu_df[
    ["mass", "radius", "orbital_period", "semi_major_axis", "star_mass", "star_radius"]
].copy()
summary = pd.DataFrame(
    {
        "mean": stats_df.mean(numeric_only=True),
        "median": stats_df.median(numeric_only=True),
        "std": stats_df.std(numeric_only=True),
        "skew": stats_df.skew(numeric_only=True),
    }
)
summary

,mean,median,std,skew
mass,1.526592,0.380000,2.644163,2.376268
radius,0.409296,0.222000,0.442455,2.744018
orbital_period,1165.143887,10.526294,42005.857581,57.675311
semi_major_axis,30.415428,0.102500,1224.655785,60.850610
star_mass,0.929399,0.940000,0.335555,1.023767
star_radius,1.360995,0.960000,2.613894,11.199827


In [17]:
positive_columns = [
    "mass",
    "radius",
    "orbital_period",
    "semi_major_axis",
    "star_mass",
    "star_radius",
]
log_ready = eu_df[positive_columns].apply(lambda col: np.log10(col.where(col > 0)))
log_ready.describe().T[["mean", "std", "min", "max"]]

,mean,std,min,max
mass,-0.601754,0.989029,-3.721246,1.111733
radius,-0.569533,0.382146,-5.698970,0.838849
orbital_period,1.224539,0.888704,-1.611828,6.435032
semi_major_axis,-0.760384,0.774372,-2.908474,4.906761
star_mass,-0.067734,0.199827,-1.958607,0.740363
star_radius,-0.004271,0.291264,-4.844664,1.708421
